# 웹 크롤링 + Embedding + Chroma DB 기반 RAG FAQ

## 주제
Python 공식 문서(docs.python.org)를 직접 크롤링해 원천 데이터를 만들고, 문서를 청크로 나누어 임베딩한 뒤 Chroma DB에 저장한다. 사용자 질문과 가장 가까운 문서를 검색하고, 검색된 문맥만 근거로 FAQ 답변을 생성한다.

## 전체 흐름
1. 웹 크롤링으로 원천 데이터 확보
2. HTML 정제 및 청크 분할
3. Embedding 생성 및 Chroma DB 저장
4. 유사도 검색(Retrieval)
5. 검색 문맥 기반 FAQ 생성
6. 검색 품질 평가 및 결과 저장

## 사용 기술
- `requests` + `BeautifulSoup`: 웹 크롤링 / HTML 파싱
- `chromadb.PersistentClient`: 로컬 벡터 DB
- `chromadb.utils.embedding_functions.DefaultEmbeddingFunction`: API 키 없이 로컬에서 동작하는 임베딩(`all-MiniLM-L6-v2`)

## 산출물 위치
- `source_data/`: 크롤링한 원천 데이터(문서 본문 텍스트)
- `result_data/`: 노트북이 만들어 낸 결과물(크롤링 목록, FAQ, 검색 평가표, 벡터 DB)

> 주의: 수집 시점과 네트워크 상태에 따라 문서 내용이 바뀔 수 있으므로 URL, 수집 시각, 청크 메타데이터를 함께 저장한다.

## 0. 환경 준비

실습 환경에 필요한 패키지가 없다면 아래 주석을 해제해 설치한다.

In [ ]:
# !pip -q install requests beautifulsoup4 chromadb pandas

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import json
import re
import sys

import pandas as pd
import requests
from bs4 import BeautifulSoup
import chromadb
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

# Windows 콘솔 등 비 UTF-8 환경에서도 출력이 깨지지 않도록 설정한다.
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass

def find_base_dir():
    """노트북 파일이 놓인 폴더를 작업 기준 경로로 삼는다."""
    candidates = [Path.cwd(), Path.cwd() / "rag", Path.cwd() / "hw" / "김재학" / "rag"]
    for candidate in candidates:
        if (candidate / "rag_faq.ipynb").exists():
            return candidate
    return Path.cwd()

BASE_DIR = find_base_dir()

# 크롤링한 원천 데이터는 source_data, 노트북이 만들어 낸 결과물은 result_data에 모은다.
SOURCE_DIR = BASE_DIR / "source_data"
RESULT_DIR = BASE_DIR / "result_data"
for directory in (SOURCE_DIR, RESULT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DB_DIR = RESULT_DIR / "chroma_db"
HEADERS = {"User-Agent": "HateSlop-RAG-assignment/1.0 (educational use)"}

print("작업 경로:", BASE_DIR.resolve())
print("원천 데이터 폴더:", SOURCE_DIR.resolve())
print("결과물 폴더:", RESULT_DIR.resolve())
print("Chroma 버전:", chromadb.__version__)

작업 경로: C:\Users\SEC\Desktop\HateSlop\HateSlop_week4-2\hw\김재학\rag
원천 데이터 폴더: C:\Users\SEC\Desktop\HateSlop\HateSlop_week4-2\hw\김재학\rag\source_data
결과물 폴더: C:\Users\SEC\Desktop\HateSlop\HateSlop_week4-2\hw\김재학\rag\result_data
Chroma 버전: 1.5.9


## 1. 웹 크롤링으로 원천 데이터 확보

공식 문서는 Sphinx로 만들어져 본문이 `div[role="main"]` 안에 들어 있다. 본문만 남기고 네비게이션, 사이드바, 검색창, 스크립트를 제거해 검색에 불필요한 텍스트를 없앤다.

In [2]:
URLS = [
    "https://docs.python.org/3/tutorial/datastructures.html",
    "https://docs.python.org/3/tutorial/errors.html",
    "https://docs.python.org/3/faq/programming.html",
    "https://docs.python.org/3/faq/general.html",
    "https://docs.python.org/3/tutorial/index.html",
]

def crawl_page(url):
    """공식 문서 한 페이지를 요청해 본문 텍스트만 추출한다."""
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    # 서버가 charset을 명시하지 않으면 requests가 latin-1로 해석해 한글/기호가 깨진다.
    if not response.encoding or response.encoding.lower() in ("iso-8859-1", "ascii"):
        response.encoding = response.apparent_encoding or "utf-8"
    soup = BeautifulSoup(response.text, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
        tag.decompose()
    for selector in ["div.related", "div.sphinxsidebar", "div.inline-search"]:
        for tag in soup.select(selector):
            tag.decompose()
    main = soup.find("div", {"role": "main"}) or soup.find("main") or soup.find("body")
    heading = main.find("h1") or soup.find("h1") or soup.find("title")
    title = heading.get_text(" ", strip=True) if heading else url
    text = main.get_text(" ", strip=True)

    # 문단 기호, 줄바꿈 방지 공백, 스마트 인용부호를 정리해 읽기 쉽게 만든다.
    def cleanup(value):
        for code, replacement in [(0xB6, " "), (0xA0, " "), (0x2019, "'"), (0x201C, '"'), (0x201D, '"'), (0x2014, "-")]:
            value = value.replace(chr(code), replacement)
        return re.sub(r"\s+", " ", value).strip()

    return {
        "url": url,
        "title": cleanup(title),
        "text": cleanup(text),
        "crawled_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }

raw_pages = [crawl_page(url) for url in URLS]
for page in raw_pages:
    print(f"- {page['title']}: {len(page['text']):,}자")
    print(f"  {page['url']}")

- 5. Data Structures: 22,808자
  https://docs.python.org/3/tutorial/datastructures.html
- 8. Errors and Exceptions: 21,093자
  https://docs.python.org/3/tutorial/errors.html
- Programming FAQ: 68,327자
  https://docs.python.org/3/faq/programming.html
- General Python FAQ: 17,405자
  https://docs.python.org/3/faq/general.html
- The Python Tutorial: 6,118자
  https://docs.python.org/3/tutorial/index.html


In [3]:
# 크롤링한 원천 데이터를 source_data 폴더에 저장한다. (README의 '크롤링 결과물' 요구사항)
index_rows = []
for page in raw_pages:
    file_name = page["url"].rstrip("/").split("/")[-1].replace(".html", "") + ".txt"
    (SOURCE_DIR / file_name).write_text(page["text"], encoding="utf-8")
    index_rows.append({
        "title": page["title"],
        "url": page["url"],
        "chars": len(page["text"]),
        "crawled_at": page["crawled_at"],
        "file": f"source_data/{file_name}",
    })
crawl_index = pd.DataFrame(index_rows)
crawl_index.to_csv(RESULT_DIR / "crawled_index.csv", index=False, encoding="utf-8-sig")
print(crawl_index.to_string(index=False))
print("\n수집 원문 예시:\n", raw_pages[3]["text"][:500])

                   title                                                    url  chars                crawled_at                           file
      5. Data Structures https://docs.python.org/3/tutorial/datastructures.html  22808 2026-10-07T12:47:48+00:00 source_data/datastructures.txt
8. Errors and Exceptions         https://docs.python.org/3/tutorial/errors.html  21093 2026-10-07T12:47:48+00:00         source_data/errors.txt
         Programming FAQ         https://docs.python.org/3/faq/programming.html  68327 2026-10-07T12:47:49+00:00    source_data/programming.txt
      General Python FAQ             https://docs.python.org/3/faq/general.html  17405 2026-10-07T12:47:50+00:00        source_data/general.txt
     The Python Tutorial          https://docs.python.org/3/tutorial/index.html   6118 2026-10-07T12:47:50+00:00          source_data/index.txt

수집 원문 예시:
 General Python FAQ General Information What is Python? Python is an interpreted, interactive, object-oriented programming la

## 2. 문서 정제와 청크 분할

문서 전체를 하나로 임베딩하면 검색 단위가 너무 커진다. 약 900자 단위로 나누되 120자를 겹치게 하여 문장이 경계에서 잘리는 문제를 완화한다. 각 청크에는 URL, 제목, 청크 번호, 수집 시각을 메타데이터로 붙인다.

In [4]:
def split_text(text, chunk_size=900, overlap=120):
    """문장 경계를 고려해 일정 길이로 텍스트를 자른다."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            boundary = text.rfind(". ", start + chunk_size // 2, end)
            if boundary > start:
                end = boundary + 1
        chunk = text[start:end].strip()
        if len(chunk) >= 80:
            chunks.append(chunk)
        if end >= len(text):
            break
        start = max(end - overlap, start + 1)
    return chunks

documents, metadatas, ids = [], [], []
for page_index, page in enumerate(raw_pages):
    for chunk_index, chunk in enumerate(split_text(page["text"])):
        documents.append(chunk)
        metadatas.append({
            "url": page["url"],
            "title": page["title"],
            "chunk_index": chunk_index,
            "crawled_at": page["crawled_at"],
        })
        ids.append(f"page{page_index}_chunk{chunk_index}")

print("페이지 수:", len(raw_pages))
print("생성된 청크 수:", len(documents))
print("청크 예시:\n", documents[0][:300])

페이지 수: 5
생성된 청크 수: 197
청크 예시:
 5. Data Structures This chapter describes some things you've learned about already in more detail, and adds some new things as well. 5.1. More on Lists The list data type has some more methods. Here are all of the methods of list objects: list. append ( value , / ) Add an item to the end of the list


## 3. Embedding 생성과 Chroma DB 저장

Chroma 컬렉션에 임베딩 함수를 연결하면 `add()` 시점에 자동으로 임베딩이 만들어져 저장된다. 반복 실행 시 결과가 섞이지 않도록 같은 이름의 컬렉션을 먼저 삭제하고 다시 만든다. 거리 계산은 코사인 유사도를 사용한다.

In [5]:
embedding_function = DefaultEmbeddingFunction()
chroma_client = chromadb.PersistentClient(path=str(DB_DIR))

try:
    chroma_client.delete_collection("python_docs")
except Exception:
    pass

collection = chroma_client.get_or_create_collection(
    name="python_docs",
    embedding_function=embedding_function,
    metadata={"description": "Python official documentation chunks", "hnsw:space": "cosine"},
)

collection.add(ids=ids, documents=documents, metadatas=metadatas)
print("컬렉션 이름:", collection.name)
print("저장된 문서 수:", collection.count())
print("DB 경로:", DB_DIR.resolve())

컬렉션 이름: python_docs
저장된 문서 수: 197
DB 경로: C:\Users\SEC\Desktop\HateSlop\HateSlop_week4-2\hw\김재학\rag\result_data\chroma_db


## 4. 유사도 검색

질문을 임베딩해 Chroma에서 가장 가까운 청크를 가져온다. 문서 본문과 출처, 거리(distance)를 함께 반환해 답변의 근거를 확인할 수 있게 한다.

In [6]:
def retrieve(question, n_results=3):
    result = collection.query(
        query_texts=[question],
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )
    return [
        {"document": document, "metadata": metadata, "distance": distance}
        for document, metadata, distance in zip(
            result["documents"][0], result["metadatas"][0], result["distances"][0]
        )
    ]

demo_question = "What is the difference between lists and tuples?"
for rank, hit in enumerate(retrieve(demo_question), 1):
    print(f"[{rank}] distance={hit['distance']:.4f} | {hit['metadata']['title']}")
    print(f"    {hit['metadata']['url']}")
    print("   ", hit["document"][:220], "\n")

[1] distance=0.2644 | 5. Data Structures
    https://docs.python.org/3/tutorial/datastructures.html
    eted correctly; they may be input with or without surrounding parentheses, although often parentheses are necessary anyway (if the tuple is part of a larger expression). It is not possible to assign to the individual ite 

[2] distance=0.4198 | Programming FAQ
    https://docs.python.org/3/faq/programming.html
    assignment is executed, and its return value is what gets used in the assignment statement; and (b) for lists, __iadd__() is equivalent to calling extend() on the list and returning the list. That's why we say that for l 

[3] distance=0.4244 | Programming FAQ
    https://docs.python.org/3/faq/programming.html
    od or function and accumulate the return values in a list, a list comprehension is an elegant solution: result = [ obj . method () for obj in mylist ] result = [ function ( obj ) for obj in mylist ] To just run the metho 



### 4-1. 한국어 질의와 영어 질의 비교

Chroma 기본 임베딩 모델(`all-MiniLM-L6-v2`)은 영어 중심으로 학습되어 한국어 질의의 검색 성능이 떨어진다. 아래 셀에서 같은 의미의 질문을 언어만 바꿔 검색해 차이를 확인한다. 이 실험 때문에 이후 FAQ는 **한국어 질문 + 영어 검색 질의**를 함께 사용한다.

In [7]:
pairs = [
    ("파이썬은 어떤 언어인가요?", "What is Python?"),
    ("문자열을 숫자로 바꾸려면?", "How do I convert a string to a number?"),
]
comparison_rows = []
for ko_query, en_query in pairs:
    for label, query in [("한국어 질의", ko_query), ("영어 질의", en_query)]:
        top = retrieve(query, 1)[0]
        comparison_rows.append({
            "구분": label,
            "질의": query,
            "최상위 문서": top["metadata"]["title"],
            "distance": round(top["distance"], 4),
        })
print(pd.DataFrame(comparison_rows).to_string(index=False))

    구분                                     질의             최상위 문서  distance
한국어 질의                         파이썬은 어떤 언어인가요? General Python FAQ    0.8462
 영어 질의                        What is Python? General Python FAQ    0.2500
한국어 질의                         문자열을 숫자로 바꾸려면? General Python FAQ    0.8674
 영어 질의 How do I convert a string to a number?    Programming FAQ    0.3885


## 5. 검색 문맥 기반 FAQ 생성

생성형 모델 없이도 RAG의 핵심인 "검색된 문맥으로 답하기"를 재현한다. 공식 FAQ 문서는 질문이 소제목으로 들어 있으므로, 검색된 청크에서 질문 소제목을 찾아 **그 바로 뒤 문장**을 답변으로 사용한다. 소제목이 없으면 가장 가까운 청크에서 질의 키워드와 겹치는 문장을 고른다. 검색 문맥의 문장만 사용하므로 문서에 없는 내용을 지어낼 위험이 낮다.

In [8]:
STOPWORDS = set(
    "the a an is are be to of in for and or how what why does do can i my you it as with that this these those".split()
)

# 한국어 FAQ 질문, 검색에 쓸 영어 질의, 검증용 기대 출처를 함께 정의한다.
# 공식 FAQ 문서는 질문이 소제목으로 들어 있으므로, 영어 질의를 문서 소제목과 동일하게 맞추면
# 검색 정확도와 답변 추출 품질이 모두 좋아진다.
FAQ_SPECS = [
    {"ko": "파이썬은 어떤 언어인가요?", "en": "What is Python?", "source": "faq/general.html"},
    {"ko": "파이썬은 원래 왜 만들어졌나요?", "en": "Why was Python created in the first place?", "source": "faq/general.html"},
    {"ko": "파이썬은 어디에 쓰기 좋은가요?", "en": "What is Python good for?", "source": "faq/general.html"},
    {"ko": "문자열을 숫자로 바꾸려면 어떻게 하나요?", "en": "How do I convert a string to a number?", "source": "faq/programming.html"},
    {"ko": "숫자를 문자열로 바꾸려면 어떻게 하나요?", "en": "How do I convert a number to a string?", "source": "faq/programming.html"},
    {"ko": "리스트와 튜플을 서로 바꾸려면 어떻게 하나요?", "en": "How do I convert between tuples and lists?", "source": "faq/programming.html"},
    {"ko": "리스트에서 중복을 제거하려면 어떻게 하나요?", "en": "How do you remove duplicates from a list?", "source": "faq/programming.html"},
    {"ko": "여러 모듈에서 전역 변수를 공유하려면 어떻게 하나요?", "en": "How do I share global variables across modules?", "source": "faq/programming.html"},
    {"ko": "파이썬에서 객체를 복사하려면 어떻게 하나요?", "en": "How do I copy an object in Python?", "source": "faq/programming.html"},
    # tutorial 문서는 질문 형태가 아니라 섹션 제목을 사용한다.
    {"ko": "예외 처리는 어떻게 하나요?", "en": "Handling Exceptions", "source": "tutorial/errors.html"},
]

def keywords(text):
    """영문 텍스트에서 비교적 의미 있는 단어만 남긴다."""
    return {
        word.lower()
        for word in re.findall(r"[A-Za-z][A-Za-z0-9_+-]{1,}", text)
        if word.lower() not in STOPWORDS
    }

def sentences_after_question(text, question, max_sentences):
    """문서 안에서 질문 소제목을 찾아, 그 뒤에 오는 문장을 답변 후보로 반환한다."""
    for marker in (question.strip(), question.rstrip("?").strip()):
        position = text.lower().find(marker.lower())
        if position != -1:
            tail = text[position + len(marker):]
            picked = []
            for sentence in re.split(r"(?<=[.!?])\s+", tail):
                sentence = sentence.strip()
                if len(sentence) < 50 or ">>>" in sentence:
                    continue
                picked.append(sentence)
                if len(picked) == max_sentences:
                    break
            if picked:
                return picked
    return []

def extract_answer(query_en, n_results=6, max_sentences=2):
    hits = retrieve(query_en, n_results)

    # 1순위: 문서에 질문이 소제목으로 있는 경우, 질문 바로 뒤 문장을 답변으로 쓴다.
    for hit in hits:
        picked = sentences_after_question(hit["document"], query_en, max_sentences)
        if picked:
            return {
                "answer": " ".join(picked),
                "sources": [hit["metadata"]["url"]],
                "hits": hits,
                "method": "질문 소제목 뒤 문장 추출",
            }

    # 2순위: 소제목이 없으면 가장 가까운 청크에서 질의 키워드와 겹치는 문장을 고른다.
    top = hits[0]
    query_terms = keywords(query_en)
    scored = []
    for sentence in re.split(r"(?<=[.!?])\s+", top["document"]):
        sentence = sentence.strip()
        if len(sentence) < 60 or ">>>" in sentence:
            continue
        scored.append((len(query_terms & keywords(sentence)), sentence))
    scored.sort(reverse=True)
    picked = [sentence for _, sentence in scored[:max_sentences]]
    if not picked:
        picked = [
            sentence.strip()
            for sentence in re.split(r"(?<=[.!?])\s+", top["document"])
            if len(sentence.strip()) >= 60
        ][:max_sentences]
    return {
        "answer": " ".join(picked),
        "sources": [top["metadata"]["url"]],
        "hits": hits,
        "method": "키워드 기반 문장 추출",
    }

faq_results = []
for spec in FAQ_SPECS:
    extracted = extract_answer(spec["en"])
    faq_results.append({
        "question_ko": spec["ko"],
        "question_en": spec["en"],
        "answer": extracted["answer"],
        "sources": extracted["sources"],
        "method": extracted["method"],
        "top_distance": round(extracted["hits"][0]["distance"], 4),
    })

for number, item in enumerate(faq_results, 1):
    print(f"[FAQ {number}] {item['question_ko']}")
    print(f"  검색 질의: {item['question_en']}  ({item['method']})")
    print(f"  답변: {item['answer'][:350]}")
    print(f"  출처: {item['sources'][0]}\n")

[FAQ 1] 파이썬은 어떤 언어인가요?
  검색 질의: What is Python?  (질문 소제목 뒤 문장 추출)
  답변: Python is an interpreted, interactive, object-oriented programming language. It incorporates modules, exceptions, dynamic typing, very high level dynamic data types, and classes.
  출처: https://docs.python.org/3/faq/general.html

[FAQ 2] 파이썬은 원래 왜 만들어졌나요?
  검색 질의: Why was Python created in the first place?  (질문 소제목 뒤 문장 추출)
  답변: Here's a very brief summary of what started it all, written by Guido van Rossum: I had extensive experience with implementing an interpreted language in the ABC group at CWI, and from working with this group I had learned a lot about language design. This is the origin of many Python features, including the use of indentation for statement grouping
  출처: https://docs.python.org/3/faq/general.html

[FAQ 3] 파이썬은 어디에 쓰기 좋은가요?
  검색 질의: What is Python good for?  (질문 소제목 뒤 문장 추출)
  답변: Python is a high-level general-purpose programming language that can be applied to many different classes of pro

## 6. (선택) LLM으로 한국어 답변 생성

검색 문맥을 그대로 LLM에 전달해 자연스러운 한국어 답변을 만들 수 있다. 이 셀은 기본값이 `False`라서 `OPENAI_API_KEY` 없이도 노트북이 끝까지 실행된다. 사용하려면 `USE_LLM_ANSWER = True`로 바꾸고 `.env`에 키를 설정한다. 문맥에 없는 내용은 답하지 않도록 프롬프트에서 제약을 준다.

In [9]:
USE_LLM_ANSWER = True  # True로 바꾸면 LLM을 이용한 답변을 추가로 생성한다.

def llm_answer(question_ko, contexts):
    from openai import OpenAI

    client = OpenAI()
    context_text = "\n\n".join(contexts)
    prompt = (
        "아래 [문서]에 있는 내용만 사용해서 질문에 한국어로 3문장 이내로 답하세요. "
        "문서에 없는 내용은 추측하지 말고 '문서에서 찾을 수 없습니다'라고 답하세요.\n\n"
        f"[문서]\n{context_text}\n\n[질문]\n{question_ko}"
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    return response.choices[0].message.content.strip()

if USE_LLM_ANSWER:
    try:
        from dotenv import load_dotenv

        load_dotenv()
        for item in faq_results:
            contexts = [hit["document"] for hit in retrieve(item["question_en"], 3)]
            item["llm_answer"] = llm_answer(item["question_ko"], contexts)
            print(f"Q. {item['question_ko']}\nA. {item['llm_answer']}\n")
    except Exception as error:
        print("LLM 답변 생략:", error)
else:
    print("USE_LLM_ANSWER = False 이므로 추출형 답변만 사용합니다.")

Q. 파이썬은 어떤 언어인가요?
A. 파이썬은 해석되고 상호작용적인 객체 지향 프로그래밍 언어입니다. 이 언어는 모듈, 예외, 동적 타이핑, 고수준 동적 데이터 타입 및 클래스를 포함하고 있으며, 절차적 및 함수형 프로그래밍과 같은 여러 프로그래밍 패러다임을 지원합니다. 파이썬은 매우 명확한 문법과 함께 놀라운 기능을 결합하고 있습니다.

Q. 파이썬은 원래 왜 만들어졌나요?
A. 파이썬은 Guido van Rossum이 ABC 언어에서의 경험을 바탕으로 언어 설계에 대한 많은 것을 배운 후, 그 언어의 확장성 부족에 대한 불만을 해결하기 위해 만들어졌습니다. 그는 ABC 언어의 여러 기능을 좋아했지만, 그 언어를 확장할 수 없다는 점이 큰 문제라고 느꼈습니다. 이러한 배경에서 파이썬의 여러 기능이 발전하게 되었습니다.

Q. 파이썬은 어디에 쓰기 좋은가요?
A. 파이썬은 다양한 문제에 적용할 수 있는 고급 범용 프로그래밍 언어입니다. 문자열 처리, 인터넷 프로토콜, 소프트웨어 공학, 운영 체제 인터페이스 등 여러 분야를 아우르는 대규모 표준 라이브러리를 제공합니다. 또한, 다양한 서드파티 확장도 이용할 수 있습니다.

Q. 문자열을 숫자로 바꾸려면 어떻게 하나요?
A. 문자열을 숫자로 바꾸려면 정수의 경우 내장형 int() 생성자를 사용하면 됩니다. 예를 들어, int('144')는 144와 같습니다. 부동 소수점 숫자로 변환하려면 float()를 사용하면 되며, 예를 들어 float('144')는 144.0과 같습니다.

Q. 숫자를 문자열로 바꾸려면 어떻게 하나요?
A. 숫자를 문자열로 바꾸려면 내장형 타입 생성자인 str()을 사용하면 됩니다. 예를 들어, 숫자 144를 문자열 '144'로 변환하려면 str(144)를 사용하면 됩니다. 또한, 16진수나 8진수 표현이 필요하면 각각 hex()나 oct() 함수를 사용할 수 있습니다.

Q. 리스트와 튜플을 서로 바꾸려면 어떻게 하나요?
A. 리스트와 튜플을 서로 바꾸려면 `tuple(seq)` 타입 생성

## 7. FAQ 결과 저장

생성한 FAQ와 검색 근거를 파일로 저장해 제출물에서 확인할 수 있게 한다.

In [10]:
faq_json_path = RESULT_DIR / "python_docs_faq.json"
faq_csv_path = RESULT_DIR / "python_docs_faq.csv"

with faq_json_path.open("w", encoding="utf-8") as file:
    json.dump(faq_results, file, ensure_ascii=False, indent=2)

pd.DataFrame(faq_results).to_csv(faq_csv_path, index=False, encoding="utf-8-sig")
print("저장 완료:")
print(" -", faq_json_path.resolve())
print(" -", faq_csv_path.resolve())

저장 완료:
 - C:\Users\SEC\Desktop\HateSlop\HateSlop_week4-2\hw\김재학\rag\result_data\python_docs_faq.json
 - C:\Users\SEC\Desktop\HateSlop\HateSlop_week4-2\hw\김재학\rag\result_data\python_docs_faq.csv


## 8. 검색 품질 평가

각 질문마다 정답이 들어 있는 문서를 미리 지정해 두고, Top-1 / Top-3 검색 결과에 그 문서가 포함되는지 확인한다. 크롤링 → 청킹 → 임베딩 → 검색 파이프라인이 정상 동작하는지 확인하는 지표로 사용한다.

In [11]:
eval_rows = []
for spec in FAQ_SPECS:
    hits = retrieve(spec["en"], 3)
    urls = [hit["metadata"]["url"] for hit in hits]
    eval_rows.append({
        "FAQ": spec["ko"],
        "기대 문서": spec["source"],
        "Top1 일치": spec["source"] in urls[0],
        "Top3 포함": any(spec["source"] in url for url in urls),
        "Top1 distance": round(hits[0]["distance"], 4),
    })

eval_df = pd.DataFrame(eval_rows)
eval_df.to_csv(RESULT_DIR / "retrieval_eval.csv", index=False, encoding="utf-8-sig")
print(eval_df.to_string(index=False))
print(f"\nTop-1 정확도: {eval_df['Top1 일치'].mean():.0%}")
print(f"Top-3 포함률: {eval_df['Top3 포함'].mean():.0%}")

                          FAQ                기대 문서  Top1 일치  Top3 포함  Top1 distance
               파이썬은 어떤 언어인가요?     faq/general.html     True     True         0.2500
            파이썬은 원래 왜 만들어졌나요?     faq/general.html     True     True         0.2641
            파이썬은 어디에 쓰기 좋은가요?     faq/general.html     True     True         0.2999
       문자열을 숫자로 바꾸려면 어떻게 하나요? faq/programming.html     True     True         0.3885
       숫자를 문자열로 바꾸려면 어떻게 하나요? faq/programming.html     True     True         0.3857
    리스트와 튜플을 서로 바꾸려면 어떻게 하나요? faq/programming.html    False     True         0.3472
     리스트에서 중복을 제거하려면 어떻게 하나요? faq/programming.html     True     True         0.3693
여러 모듈에서 전역 변수를 공유하려면 어떻게 하나요? faq/programming.html     True     True         0.3538
     파이썬에서 객체를 복사하려면 어떻게 하나요? faq/programming.html     True     True         0.3872
              예외 처리는 어떻게 하나요? tutorial/errors.html     True     True         0.3067

Top-1 정확도: 90%
Top-3 포함률: 100%


## 9. 회고 및 한계

1. 실제 웹(공식 문서)을 크롤링해 원천 데이터를 확보하고, 본문만 남기는 정제 과정을 거쳤다.
2. 청크 단위 임베딩과 Chroma 영구 저장으로 재실행 가능한 벡터 DB를 구성했다.
3. 검색 문맥의 문장만 조합하는 추출형 FAQ는 환각이 없지만 문장이 자연스럽지 않고, 문서가 길면 핵심이 아닌 문장이 뽑힐 수 있다. 6번 셀의 LLM 답변 생성으로 보완할 수 있다.
4. 기본 임베딩(`all-MiniLM-L6-v2`)이 영어 중심이라 한국어 질의의 검색 성능이 낮다. 한국어 질의를 그대로 쓰려면 `sentence-transformers`의 다국어 모델이나 OpenAI `text-embedding-3-small` 같은 다국어 임베딩으로 교체해야 한다.
5. 문서가 갱신되면 같은 질문의 검색 결과가 달라질 수 있어 URL과 수집 시각을 메타데이터로 남겼다.

### 생성되는 파일
- `rag_faq.ipynb`: 전체 실습 흐름(제출물)
- `source_data/*.txt`: 크롤링한 원천 데이터(문서 본문)
- `result_data/crawled_index.csv`: 크롤링한 문서 목록(URL/수집 시각/글자 수)
- `result_data/python_docs_faq.json`, `result_data/python_docs_faq.csv`: 생성된 FAQ
- `result_data/retrieval_eval.csv`: 검색 품질 평가 결과
- `result_data/chroma_db/`: Chroma 벡터 DB(노트북 재실행 시 자동 생성)